# Module 1: Let's Understand Search

## What you will do

1. Watch keyword search miss documents that plainly answer the query.
2. Turn text into a vector and look at what actually comes back.
3. Score meaning with cosine similarity, first by hand, then with the library helper.
4. Re-run the Section 1 failures semantically and watch them pass.
5. Meet the SKU problem, where similarity on its own is not enough.

**Tip:** every cell below is already run, so you can read this straight through. Run it yourself and you should see the same numbers.

Companion notebook to the [Module 1 lesson](https://qdrant.tech/course/beginners/module-1/).

## Setup

Module 1 stays deliberately small: one embedding model, no vector database yet. `sentence-transformers` gives us `all-MiniLM-L6-v2`, a 384-dimension model fast enough to run on a free Colab CPU. Qdrant arrives in Module 2.

In [ ]:
!pip install -q sentence-transformers

## 1. The Problem: Why Keyword Search Struggles

Traditional search matches exact words. If the query string appears in the document it is a hit, and if it does not it is a miss, however close the meaning is.

Here is the whole of keyword search, in three lines.

In [ ]:
documents = [
    "automobile maintenance guide",
    "affordable airfare to New York",
    "apple harvest season guide",
]

def keyword_search(query, docs):
    """All of keyword search: does the query string appear verbatim?"""
    return [d for d in docs if query.lower() in d.lower()]

for query in ["car repair", "cheap flights NYC", "Apple stock"]:
    hits = keyword_search(query, documents)
    print(f"{query!r:22} -> {hits if hits else 'NO MATCH'}")

'car repair'           -> NO MATCH
'cheap flights NYC'    -> NO MATCH
'Apple stock'          -> NO MATCH


Three queries. Three documents that answer them. Zero hits.

Nothing is broken. The engine is doing exactly what it was asked: comparing characters. That single limitation shows up in four ways.

- **Synonyms.** "car" and "automobile" mean the same thing and share no letters.
- **Paraphrasing.** "cheap flights" and "affordable airfare" are the same intent in different words.
- **Polysemy.** "apple" is a company, a fruit, and a record label. Characters cannot tell you which one was meant.
- **Word order.** "dog bites man" and "man bites dog" contain identical words.

Improvements to keyword search, inverted indexes for speed, BM25 for ranking, stemming, and fuzzy matching, all raise the ceiling without moving it. They still work on words. You could hard-code that "car" means "automobile", then do it again for the next pair, and the next. You cannot hard-code a language.

So semantic search changes the question. Not *does this document contain the same words*, but *does this document mean the same thing*.

## 2. How It Works: Embeddings

An embedding model takes a piece of text and returns a fixed-length list of floating-point numbers. Similar meanings produce vectors that sit close together in that space.

In [ ]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("all-MiniLM-L6-v2")

query_vec = model.encode("car repair")
doc_vec = model.encode("automobile maintenance")

print("dimensions:", len(query_vec))
print("first five:", query_vec[:5])

dimensions: 384
first five: [-0.12456685  0.03955904  0.08732592 -0.02361241 -0.07054088]


384 numbers. That is the vector.

Do not go looking for the dimension that stores colour, or urgency. Nobody designed these. The model learned them from data, and meaning lives in the combination of all 384 rather than in any single one. A 384-dimension model has 384 such aspects, and only the whole set together encodes meaning.

## 3. Comparing Meaning: Cosine Similarity

Cosine similarity measures the angle between two vectors and ignores their length. Vectors pointing the same way score near 1. Unrelated directions score near 0.

The formula is short enough to write out, which is worth doing once so the library call stops being magic.

In [ ]:
import numpy as np

def cosine_by_hand(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

by_hand = cosine_by_hand(query_vec, doc_vec)
by_library = util.cos_sim(query_vec, doc_vec).item()

print(f"by hand:    {by_hand:.6f}")
print(f"cos_sim:    {by_library:.6f}")
print(f"agree:      {np.isclose(by_hand, by_library)}")

by hand:    0.733420
cos_sim:    0.733420
agree:      True


Roughly 0.73 for two phrases with no words in common. That number is the whole point of the module.

Now three pairs at once: a synonym pair, a paraphrase pair, and a pair that has nothing to do with each other.

In [ ]:
pairs = [
    ("car repair", "automobile maintenance"),                    # synonyms
    ("cheap flights to New York", "affordable airfare to NYC"),  # paraphrase
    ("cheap flights to New York", "best pizza in Chicago"),      # unrelated
]

for query, document in pairs:
    score = util.cos_sim(model.encode(query), model.encode(document)).item()
    print(f"{score:.3f}  |  {query!r}  vs  {document!r}")

0.733  |  'car repair'  vs  'automobile maintenance'
0.821  |  'cheap flights to New York'  vs  'affordable airfare to NYC'
0.332  |  'cheap flights to New York'  vs  'best pizza in Chicago'


The first two pairs share little or no vocabulary and both score high. That is the synonym and the paraphrase that keyword search missed in Section 1. The third scores far lower, so the model is separating meaning rather than matching surface words.

Which means we can go back and re-run those three failures.

In [ ]:
# Same three documents as Section 1, plus a second "apple" sense so the
# polysemy case has somewhere correct to land.
documents = [
    "automobile maintenance guide",
    "affordable airfare to New York",
    "apple harvest season guide",
    "technology shares closed higher today",
]

doc_vecs = model.encode(documents)

for query in ["car repair", "cheap flights NYC", "Apple stock"]:
    sims = util.cos_sim(model.encode(query), doc_vecs)[0]
    ranked = sorted(zip(sims.tolist(), documents), reverse=True)
    print(f"{query!r}")
    for score, doc in ranked:
        print(f"   {score:6.3f}  {doc}")
    print()

'car repair'
    0.609  automobile maintenance guide
    0.065  apple harvest season guide
    0.060  technology shares closed higher today
   -0.032  affordable airfare to New York

'cheap flights NYC'
    0.800  affordable airfare to New York
    0.117  apple harvest season guide
    0.080  technology shares closed higher today
   -0.021  automobile maintenance guide

'Apple stock'
    0.401  technology shares closed higher today
    0.374  apple harvest season guide
    0.145  affordable airfare to New York
    0.048  automobile maintenance guide



Every query now finds its document, and each one was a total miss under keyword search.

Two things in that output are worth pausing on.

**Negative scores exist.** `car repair` against `affordable airfare to New York` comes out slightly below zero. Cosine similarity ranges from -1 to 1 in principle, where -1 means the vectors point in opposite directions. With normalized text models these stay small and near zero, so treat a value like -0.03 as "unrelated" rather than "opposite".

**The polysemy win is narrow.** `Apple stock` prefers the technology document over the fruit document, but only by about 0.03. The model got the sense right, and it was close. Hold on to that, because Section 4 is built on exactly this kind of narrow gap.

### Where dense search struggles too

The four failure modes were framed as keyword problems, but word order is not solved by switching to embeddings. Compare a sentence with its meaning reversed, and with a genuine paraphrase.

In [ ]:
reversed_pair = ("dog bites man", "man bites dog")
paraphrase_pair = ("dog bites man", "a canine attacked a person")

for a, b in [reversed_pair, paraphrase_pair]:
    print(f"{util.cos_sim(model.encode(a), model.encode(b)).item():.3f}  |  {a!r}  vs  {b!r}")

0.907  |  'dog bites man'  vs  'man bites dog'
0.570  |  'dog bites man'  vs  'a canine attacked a person'


The reversed sentence scores higher than the correct paraphrase.

The model rates *opposite meaning, identical words* as more similar than *same meaning, different words*. Embeddings shifted the problem rather than removing it, and that is worth knowing before you trust similarity with anything important.

## 4. Why Similarity Alone Is Not Enough

Here is the failure that costs real money. A user types an exact product code and wants that product.

In [ ]:
skus = ["SKU-48290", "SKU-48291", "SKU-48292", "SKU-48293"]
catalog = [f"{sku} replacement part, ships in 2 days" for sku in skus]

query = model.encode("SKU-48291 issue")
sims = util.cos_sim(query, model.encode(catalog))[0]

scored = sorted(zip(sims.tolist(), skus), reverse=True)
for score, sku in scored:
    marker = "  <-- the one they asked for" if sku == "SKU-48291" else ""
    print(f"{score:.3f}  {sku}{marker}")

print()
print(f"spread between best and worst: {scored[0][0] - scored[-1][0]:.3f}")

0.626  SKU-48290
0.624  SKU-48291  <-- the one they asked for
0.616  SKU-48293
0.610  SKU-48292

spread between best and worst: 0.016


The requested SKU is not first, and look at the spread: all four codes land within a couple of hundredths of each other.

That is not a bug in the model. Those strings are nearly identical, so their vectors are nearly identical, and the model has no idea that the final digit is the only part that matters. Semantically they are the same thing. Commercially, three of the four are the wrong product.

No amount of model tuning fixes this, because the model is not wrong. What is missing is a hard constraint.

In [ ]:
TARGET = "SKU-48291"

# An exact constraint on a metadata field, not a similarity score
filtered = [(score, sku) for score, sku in scored if sku == TARGET]

for score, sku in filtered:
    print(f"{score:.3f}  {sku}")

print()
print("In Qdrant that constraint is a payload filter, evaluated while the search runs:")
print('   must=[FieldCondition(key="sku", match=MatchValue(value="SKU-48291"))]')
print("You will write exactly that in Module 2.")

0.624  SKU-48291

In Qdrant that constraint is a payload filter, evaluated while the search runs:
   must=[FieldCondition(key="sku", match=MatchValue(value="SKU-48291"))]
You will write exactly that in Module 2.


### Key insight

Dense similarity finds the neighbourhood. Filters, exact matches, and payload constraints find the right point inside it. Production search needs both, which is why real systems are hybrid: dense retrieval for meaning, sparse retrieval for exact tokens, and filters constraining both.

## Your turn

One exercise, then Module 2.

Swap in your own polysemy case below. Score `"apple stock"` against both senses and see which wins, then try a word your own domain overloads: *charge*, *bank*, *lead*, *plant*. Does the model pick the sense you meant?

In [ ]:
candidates = [
    "shares of a technology company",
    "a crisp red fruit",
]

for candidate in candidates:
    score = util.cos_sim(model.encode("apple stock"), model.encode(candidate)).item()
    print(f"{score:.3f}  |  'apple stock'  vs  {candidate!r}")

0.480  |  'apple stock'  vs  'shares of a technology company'
0.225  |  'apple stock'  vs  'a crisp red fruit'


## What's next: Module 2

- What a vector is, and why it has hundreds of dimensions
- How similarity works under the hood, and when it fails
- Your first Qdrant collection: points, payloads, and your first query

[Continue to Module 2](https://qdrant.tech/course/beginners/module-2/)